(rotated-gradients)=

# Rotated Gradients

This section provides details about rotating the model gradients used in the [Regularization](#Regularization) function. Applied in conjunction with weights and sparse norms, the rotation can help reinforce structural trends, as depicted in the image below. 

```{figure} ./images/rotated_demo.png
---
scale: 30%
---
(Top) Horizontal and (bottom) vertical sections through (left) a simple dipping dyke model under smooth topography. (Center) Model recovered using the conventional smooth norm inversion. (Right) Compact model obtained with a dip rotation of $45\circ$ towards East. 
```

The rotation angles can be applied globally, or on a cell-by-cell basis to reflect local orientations, such as faults and folds.  

```{figure} ./images/local_rotation.png
---
scale: 30%
---
Example of an inversion with local rotations along the expected limbs of a folded layer.
```

## Input format

Direction information must be supplied as a `Data Group` defined on the inversion mesh. The following group types are accepted: `Dip direction & Dip`, `Strike & Dip` or as `3D vector`. 

```{figure} ./images/set_orientation.png
---
scale: 30%
---
```

The `dip` angles (degrees) are positive downward from the horizontal. The `strike` and `dip direction` angles (degrees) are positive clockwise from North. The `3D vector` vector data defines the unit normals, which are perpendicular to both the `Dip direction & Dip` and `Strike & Dip` directions. As illustrated below, all three formats define a rotated plane in 3D.

```{figure} ./images/gradient_direction.png
---
scale: 50%
--- 
```

## Background

As covered in the [Model Smoothness](smooth-ref) section, penalties on the model gradients are measured by:

$$
\mathbf{f}_i = \mathbf{G}_i (\mathbf{m} - \mathbf{m}_{ref}),
$$

where $\mathbf{G}_i$ is a finite difference operator that computes the difference in model $\mathbf{m}$ values between adjacent cells along one of the Cartesian directions: East (X), North (Y) and vertical (Z). As proposed by {cite:t}`LiOldenburg2000`, orientation information can be incorporated in the inversion by rotating the gradient operators along arbitrary directions such that 

$$
\mathbf{\hat f}_i = \mathbf{\hat G}_i (\mathbf{m} - \mathbf{m}_{ref}),
$$

where $\mathbf{\hat G}_i$ are rotated gradients along one of the $u$, $v$ and $w$-axis.

```{figure} ./images/plane_rotation.png
---
name: plane_rotation
scale: 50%
---
Rotation of the Cartesian axes along an arbitrary plane.
```

- $X \rightarrow u$ pointing down-dip
- $Y \rightarrow v$ pointing along strike
- $Z \rightarrow w$ pointing along the normal

### Special note

`Simpeg-Drivers` uses a combination of both forward and backward gradient operators to achieve better symmetry in 3D. The gradient operators are constructed based on the partial volumes intercepted by ghost cells rotated about the center of each cell. The partial volumes include all diagonal neighbours intercepted by the ghost cells.  

```{figure} ./images/partial_volumes.png
---
scale: 50%
---
Plan-view depiction of the $u,v$ forward gradient operators, made up of partial volumes from neighbouring cells.
```

The full regularization term is made up of seven terms in total

$$
\phi_m = \| \mathbf{W}_s (\mathbf{m} - \mathbf{m}_{ref}) \|_2^2 + \sum_{i = u^F,v^F,w^F} \| \mathbf{W}_i \mathbf{G}_i(\mathbf{m}) \|_2^2 + \sum_{i = u^B,v^B,w^B,} \| \mathbf{W}_i \mathbf{G}_i(\mathbf{m}) \|_2^2
$$




## Example

This section demonstrates the use of rotated gradients on a semi-realistic synthetic example. 

### Setup 

The model has been generated with the help of [Gempy](https://app.readthedocs.com/projects/mirageoscience-gempy-drivers/builds/?version__slug=stable). The model comprises a folded and faulted magnetic layer (0.5 SI) dipping 20 degrees towards East. The geometry is meant to mimic a banded-iron formation.

```{figure} ./images/fold_model.png
---
scale: 30%
---
```

From this model, we simulate residual magnetic field data along an East-West survey, 200 m line spacing and a mean terrain clearance of 150 m. For simplicity, we use a vertical inducing field with a magnitude of 50,000 nT.

```{figure} ./images/fold_model.png
---
scale: 30%
---
```

The basic components (data, model and topography) to reproduce this example can be [downloaded here]().

### Standard unconstrained inversion

As a starting point, we invert the magnetic data with standard constraints (lower bounds and reference value of 0 SI). 
The resulting smooth model shows clear breaks between the survey lines and poorly resolves the dip of the magnetic layer. The subsequent compact model further exacerbates these issues. 


### Directional constraints

To improve the continuity of the magnetic layer across lines and down-dip, we can provide trend information to the inversion from our structural control points.

We interpolate the dip and dip direction data from the `structural markers` to the inversion mesh. We use the Radial Basis Function (RBF) application, but users may want to experiment with different interpolation algorithms.

```{figure} ./images/fold_model.png
---
scale: 30%
---
```

Following the instructions presented in the [previous section](rotated-gradients), we group the interpolated data as type `Dip Direction & Dip`. Users can validate the constraint by displaying the vectors onto sections of the mesh. Once created, the orientation information is provided to the inversion. 

```{figure} ./images/invert_fault_equal.png
---
scale: 30%
---
```

The resulting model shows an improved 


We can further improve this result by decreasing the relative weight applied to the gradient penalty perpendicular (z) to the rotated plane.

```{figure} ./images/invert_fault_wz0p5.png
---
scale: 30%
---
```

